In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import time
from datetime import datetime, timedelta
import pandas as pd
import urllib.parse
import random

# 봇 탐지 방지를 위해 user agnet로 설정
options = Options()
user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'
options.add_argument(f'user-agent={user_agent}')

# Chrome 드라이버 자동 다운로드 및 설정
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 부산일보 2022-01-01 ~ 2024-12-31 기간동안 '폭염' 키워드 포함한 기사 검색 url
url = 'https://www.busan.com/search/index.php?search_string=%ED%8F%AD%EC%97%BC&page=1&cnt=0&sort=DATE/ASC&start_date=2022-01-01&end_date=2024-12-31&real_search=%ED%8F%AD%EC%97%BC&rescan_chk=0&type=B&search_writer=undefined'
driver.get(url)

In [ ]:
news_links = [] # 기사 링크를 저장할 리스트 생성

# 기사가 한 페이지당 10개 존재, 총 389페이지
# 따라서 1페이지 부터 389페이지까지 방문
for i in range(1, 390):
    # 페이지 수에 따른 url 변경 반영
    url = f'https://www.busan.com/search/index.php?search_string=%ED%8F%AD%EC%97%BC&page={i}&cnt={(i-1)*10}&sort=DATE/ASC&start_date=2022-01-01&end_date=2024-12-31&real_search=%ED%8F%AD%EC%97%BC&rescan_chk=0&type=B&search_writer=undefined'
    driver.get(url) # 페이지 이동

    time.sleep(0.8) # 페이지 로딩 대기
    
    # <ul id="search_list">로 설정된 태그 내에 <p class="title"> 태그가 존재하고 그 안에 있는 모든 <a> 태그를 찾음
    a_tags = driver.find_elements(By.XPATH, '//ul[@id="search_list"]//p[@class="title"]/a')
    page_links = [a.get_attribute('href') for a in a_tags] # 한 페이지에서 a_tags 안에 있는 href 속성 추출 (기사 링크)
    news_links.extend(page_links) # 페이지에서 추출한 링크를 전체 기사 링크에 추가

    print(len(news_links)) # 현재까지 추출된 기사 링크 수 확인

print(len(news_links)) # 전체 기사 개수
print(news_links[:5])

10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
1450
1460
1470
1480
1490
1500
1510
1520
1530
1540
1550
1560
1570
1580
1590
1600
1610
1620
1630
1640
1650
1660
1670
1680
1690
1700
1710
1720
1730
1740
1750
1760
1770
1780
1790
1800
1810
1820
1830
1840
1850
1860
1870
1880
1890
1900
1910
1920
1930
1940
1950
1960
1970
1980
1990
2000
2010
2020
2030
2040
2050
2060
2070
2080
2090
2100
2110
2120
2130
2140
2150
2160
2170
2180
2190
2200
2210
222

### 개별 기사 접근 시작

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

all_results = dict()
i = 0
err_idx = []

for link in news_links[i:500]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24] # 숫자에 해당하는 부분만 추출

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[1 / 3885] 	 0.03% 진행
[2 / 3885] 	 0.05% 진행
[3 / 3885] 	 0.08% 진행
[4 / 3885] 	 0.10% 진행
[5 / 3885] 	 0.13% 진행
[6 / 3885] 	 0.15% 진행
[7 / 3885] 	 0.18% 진행
[8 / 3885] 	 0.21% 진행
[9 / 3885] 	 0.23% 진행
[10 / 3885] 	 0.26% 진행
[11 / 3885] 	 0.28% 진행
[12 / 3885] 	 0.31% 진행
[13 / 3885] 	 0.33% 진행
[14 / 3885] 	 0.36% 진행
[15 / 3885] 	 0.39% 진행
[16 / 3885] 	 0.41% 진행
[17 / 3885] 	 0.44% 진행
[18 / 3885] 	 0.46% 진행
[19 / 3885] 	 0.49% 진행
[20 / 3885] 	 0.51% 진행
[21 / 3885] 	 0.54% 진행
[22 / 3885] 	 0.57% 진행
[23 / 3885] 	 0.59% 진행
[24 / 3885] 	 0.62% 진행
[25 / 3885] 	 0.64% 진행
[26 / 3885] 	 0.67% 진행
[27 / 3885] 	 0.69% 진행
[28 / 3885] 	 0.72% 진행
[29 / 3885] 	 0.75% 진행
[30 / 3885] 	 0.77% 진행
[31 / 3885] 	 0.80% 진행
[32 / 3885] 	 0.82% 진행
[33 / 3885] 	 0.85% 진행
[34 / 3885] 	 0.88% 진행
[35 / 3885] 	 0.90% 진행
[36 / 3885] 	 0.93% 진행
[37 / 3885] 	 0.95% 진행
[38 / 3885] 	 0.98% 진행
[39 / 3885] 	 1.00% 진행
[40 / 3885] 	 1.03% 진행
[41 / 3885] 	 1.06% 진행
[42 / 3885] 	 1.08% 진행
[43 / 3885] 	 1.11% 진행
[44 / 3885] 	 1.13% 

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 500

for link in news_links[i:1000]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[501 / 3885] 	 12.90% 진행
[502 / 3885] 	 12.92% 진행
[503 / 3885] 	 12.95% 진행
[504 / 3885] 	 12.97% 진행
[505 / 3885] 	 13.00% 진행
[506 / 3885] 	 13.02% 진행
[507 / 3885] 	 13.05% 진행
[508 / 3885] 	 13.08% 진행
[509 / 3885] 	 13.10% 진행
[510 / 3885] 	 13.13% 진행
[511 / 3885] 	 13.15% 진행
[512 / 3885] 	 13.18% 진행
[513 / 3885] 	 13.20% 진행
[514 / 3885] 	 13.23% 진행
[515 / 3885] 	 13.26% 진행
[516 / 3885] 	 13.28% 진행
[517 / 3885] 	 13.31% 진행
[518 / 3885] 	 13.33% 진행
[519 / 3885] 	 13.36% 진행
[520 / 3885] 	 13.38% 진행
[521 / 3885] 	 13.41% 진행
[522 / 3885] 	 13.44% 진행
[523 / 3885] 	 13.46% 진행
[524 / 3885] 	 13.49% 진행
[525 / 3885] 	 13.51% 진행
[526 / 3885] 	 13.54% 진행
[527 / 3885] 	 13.56% 진행
[528 / 3885] 	 13.59% 진행
[529 / 3885] 	 13.62% 진행
[530 / 3885] 	 13.64% 진행
[531 / 3885] 	 13.67% 진행
[532 / 3885] 	 13.69% 진행
[533 / 3885] 	 13.72% 진행
[534 / 3885] 	 13.75% 진행
[535 / 3885] 	 13.77% 진행
[536 / 3885] 	 13.80% 진행
[537 / 3885] 	 13.82% 진행
[538 / 3885] 	 13.85% 진행
[539 / 3885] 	 13.87% 진행
[540 / 3885] 	 13.90% 진행


In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 1000

for link in news_links[i:1500]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[1001 / 3885] 	 25.77% 진행
[1002 / 3885] 	 25.79% 진행
[1003 / 3885] 	 25.82% 진행
[1004 / 3885] 	 25.84% 진행
[1005 / 3885] 	 25.87% 진행
[1006 / 3885] 	 25.89% 진행
[1007 / 3885] 	 25.92% 진행
[1008 / 3885] 	 25.95% 진행
[1009 / 3885] 	 25.97% 진행
[1010 / 3885] 	 26.00% 진행
[1011 / 3885] 	 26.02% 진행
[1012 / 3885] 	 26.05% 진행
[1013 / 3885] 	 26.07% 진행
[1014 / 3885] 	 26.10% 진행
[1015 / 3885] 	 26.13% 진행
[1016 / 3885] 	 26.15% 진행
[1017 / 3885] 	 26.18% 진행
[1018 / 3885] 	 26.20% 진행
[1019 / 3885] 	 26.23% 진행
[1020 / 3885] 	 26.25% 진행
[1021 / 3885] 	 26.28% 진행
[1022 / 3885] 	 26.31% 진행
[1023 / 3885] 	 26.33% 진행
[1024 / 3885] 	 26.36% 진행
[1025 / 3885] 	 26.38% 진행
[1026 / 3885] 	 26.41% 진행
[1027 / 3885] 	 26.44% 진행
[1028 / 3885] 	 26.46% 진행
[1029 / 3885] 	 26.49% 진행
[1030 / 3885] 	 26.51% 진행
[1031 / 3885] 	 26.54% 진행
[1032 / 3885] 	 26.56% 진행
[1033 / 3885] 	 26.59% 진행
[1034 / 3885] 	 26.62% 진행
[1035 / 3885] 	 26.64% 진행
[1036 / 3885] 	 26.67% 진행
[1037 / 3885] 	 26.69% 진행
[1038 / 3885] 	 26.72% 진행
[1039 / 3885

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 1500

for link in news_links[i:2000]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[1501 / 3885] 	 38.64% 진행
[1502 / 3885] 	 38.66% 진행
[1503 / 3885] 	 38.69% 진행
[1504 / 3885] 	 38.71% 진행
[1505 / 3885] 	 38.74% 진행
[1506 / 3885] 	 38.76% 진행
[1507 / 3885] 	 38.79% 진행
[1508 / 3885] 	 38.82% 진행
[1509 / 3885] 	 38.84% 진행
[1510 / 3885] 	 38.87% 진행
[1511 / 3885] 	 38.89% 진행
[1512 / 3885] 	 38.92% 진행
[1513 / 3885] 	 38.94% 진행
[1514 / 3885] 	 38.97% 진행
[1515 / 3885] 	 39.00% 진행
[1516 / 3885] 	 39.02% 진행
[1517 / 3885] 	 39.05% 진행
[1518 / 3885] 	 39.07% 진행
[1519 / 3885] 	 39.10% 진행
[1520 / 3885] 	 39.12% 진행
[1521 / 3885] 	 39.15% 진행
[1522 / 3885] 	 39.18% 진행
[1523 / 3885] 	 39.20% 진행
[1524 / 3885] 	 39.23% 진행
[1525 / 3885] 	 39.25% 진행
[1526 / 3885] 	 39.28% 진행
[1527 / 3885] 	 39.31% 진행
[1528 / 3885] 	 39.33% 진행
[1529 / 3885] 	 39.36% 진행
[1530 / 3885] 	 39.38% 진행
[1531 / 3885] 	 39.41% 진행
[1532 / 3885] 	 39.43% 진행
[1533 / 3885] 	 39.46% 진행
[1534 / 3885] 	 39.49% 진행
[1535 / 3885] 	 39.51% 진행
[1536 / 3885] 	 39.54% 진행
[1537 / 3885] 	 39.56% 진행
[1538 / 3885] 	 39.59% 진행
[1539 / 3885

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 2000

for link in news_links[i:2500]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[2001 / 3885] 	 51.51% 진행
[2002 / 3885] 	 51.53% 진행
[2003 / 3885] 	 51.56% 진행
[2004 / 3885] 	 51.58% 진행
[2005 / 3885] 	 51.61% 진행
[2006 / 3885] 	 51.63% 진행
[2007 / 3885] 	 51.66% 진행
[2008 / 3885] 	 51.69% 진행
[2009 / 3885] 	 51.71% 진행
[2010 / 3885] 	 51.74% 진행
[2011 / 3885] 	 51.76% 진행
[2012 / 3885] 	 51.79% 진행
[2013 / 3885] 	 51.81% 진행
[2014 / 3885] 	 51.84% 진행
[2015 / 3885] 	 51.87% 진행
[2016 / 3885] 	 51.89% 진행
[2017 / 3885] 	 51.92% 진행
[2018 / 3885] 	 51.94% 진행
[2019 / 3885] 	 51.97% 진행
[2020 / 3885] 	 51.99% 진행
[2021 / 3885] 	 52.02% 진행
[2022 / 3885] 	 52.05% 진행
[2023 / 3885] 	 52.07% 진행
[2024 / 3885] 	 52.10% 진행
[2025 / 3885] 	 52.12% 진행
[2026 / 3885] 	 52.15% 진행
[2027 / 3885] 	 52.18% 진행
[2028 / 3885] 	 52.20% 진행
[2029 / 3885] 	 52.23% 진행
[2030 / 3885] 	 52.25% 진행
[2031 / 3885] 	 52.28% 진행
[2032 / 3885] 	 52.30% 진행
[2033 / 3885] 	 52.33% 진행
[2034 / 3885] 	 52.36% 진행
[2035 / 3885] 	 52.38% 진행
[2036 / 3885] 	 52.41% 진행
[2037 / 3885] 	 52.43% 진행
[2038 / 3885] 	 52.46% 진행
[2039 / 3885

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 2500

for link in news_links[i:3000]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[2501 / 3885] 	 64.38% 진행
[2502 / 3885] 	 64.40% 진행
[2503 / 3885] 	 64.43% 진행
[2504 / 3885] 	 64.45% 진행
[2505 / 3885] 	 64.48% 진행
[2506 / 3885] 	 64.50% 진행
[2507 / 3885] 	 64.53% 진행
[2508 / 3885] 	 64.56% 진행
[2509 / 3885] 	 64.58% 진행
[2510 / 3885] 	 64.61% 진행
[2511 / 3885] 	 64.63% 진행
[2512 / 3885] 	 64.66% 진행
[2513 / 3885] 	 64.68% 진행
[2514 / 3885] 	 64.71% 진행
[2515 / 3885] 	 64.74% 진행
[2516 / 3885] 	 64.76% 진행
[2517 / 3885] 	 64.79% 진행
[2518 / 3885] 	 64.81% 진행
[2519 / 3885] 	 64.84% 진행
[2520 / 3885] 	 64.86% 진행
[2521 / 3885] 	 64.89% 진행
[2522 / 3885] 	 64.92% 진행
[2523 / 3885] 	 64.94% 진행
[2524 / 3885] 	 64.97% 진행
[2525 / 3885] 	 64.99% 진행
[2526 / 3885] 	 65.02% 진행
[2527 / 3885] 	 65.05% 진행
[2528 / 3885] 	 65.07% 진행
[2529 / 3885] 	 65.10% 진행
[2530 / 3885] 	 65.12% 진행
[2531 / 3885] 	 65.15% 진행
[2532 / 3885] 	 65.17% 진행
[2533 / 3885] 	 65.20% 진행
[2534 / 3885] 	 65.23% 진행
[2535 / 3885] 	 65.25% 진행
[2536 / 3885] 	 65.28% 진행
[2537 / 3885] 	 65.30% 진행
[2538 / 3885] 	 65.33% 진행
[2539 / 3885

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 3000

for link in news_links[i:3500]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[3001 / 3885] 	 77.25% 진행
[3002 / 3885] 	 77.27% 진행
[3003 / 3885] 	 77.30% 진행
[3004 / 3885] 	 77.32% 진행
[3005 / 3885] 	 77.35% 진행
[3006 / 3885] 	 77.37% 진행
[3007 / 3885] 	 77.40% 진행
[3008 / 3885] 	 77.43% 진행
[3009 / 3885] 	 77.45% 진행
[3010 / 3885] 	 77.48% 진행
[3011 / 3885] 	 77.50% 진행
[3012 / 3885] 	 77.53% 진행
[3013 / 3885] 	 77.55% 진행
[3014 / 3885] 	 77.58% 진행
[3015 / 3885] 	 77.61% 진행
[3016 / 3885] 	 77.63% 진행
[3017 / 3885] 	 77.66% 진행
[3018 / 3885] 	 77.68% 진행
[3019 / 3885] 	 77.71% 진행
[3020 / 3885] 	 77.73% 진행
[3021 / 3885] 	 77.76% 진행
[3022 / 3885] 	 77.79% 진행
[3023 / 3885] 	 77.81% 진행
[3024 / 3885] 	 77.84% 진행
[3025 / 3885] 	 77.86% 진행
[3026 / 3885] 	 77.89% 진행
[3027 / 3885] 	 77.92% 진행
[3028 / 3885] 	 77.94% 진행
[3029 / 3885] 	 77.97% 진행
[3030 / 3885] 	 77.99% 진행
[3031 / 3885] 	 78.02% 진행
[3032 / 3885] 	 78.04% 진행
[3033 / 3885] 	 78.07% 진행
[3034 / 3885] 	 78.10% 진행
[3035 / 3885] 	 78.12% 진행
[3036 / 3885] 	 78.15% 진행
[3037 / 3885] 	 78.17% 진행
[3038 / 3885] 	 78.20% 진행
[3039 / 3885

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 행상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 방문해야 하는 기사수가 3885로 매우 방대 하기 때문에 500개씩 끊어서 진행 (셀 단위)

i = 3500

for link in news_links[i:]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6) 

        # 제목 추출하기
        title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
        title = title[0].text 

        # 본문 추출하기
        body = driver.find_elements(By.CLASS_NAME, 'article_content')
        body = body[0].text.replace('\n', '') 

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
        pubdate = pubdate_element[0].text
        pubdate = pubdate[5:24]

        all_results[i]['link'] = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title'] = title
        all_results[i]['body'] = body

        # 진행 상황 확인용 코드
        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
        i += 1

        # 봇 탐지 방지를 위해 0.8초에서 1.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.8, 1.5))

print(len(all_results))
print(err_idx)

[3501 / 3885] 	 90.12% 진행
[3502 / 3885] 	 90.14% 진행
[3503 / 3885] 	 90.17% 진행
[3504 / 3885] 	 90.19% 진행
[3505 / 3885] 	 90.22% 진행
[3506 / 3885] 	 90.24% 진행
[3507 / 3885] 	 90.27% 진행
[3508 / 3885] 	 90.30% 진행
[3509 / 3885] 	 90.32% 진행
[3510 / 3885] 	 90.35% 진행
[3511 / 3885] 	 90.37% 진행
[3512 / 3885] 	 90.40% 진행
[3513 / 3885] 	 90.42% 진행
[3514 / 3885] 	 90.45% 진행
[3515 / 3885] 	 90.48% 진행
[3516 / 3885] 	 90.50% 진행
[3517 / 3885] 	 90.53% 진행
[3518 / 3885] 	 90.55% 진행
[3519 / 3885] 	 90.58% 진행
[3520 / 3885] 	 90.60% 진행
[3521 / 3885] 	 90.63% 진행
[3522 / 3885] 	 90.66% 진행
[3523 / 3885] 	 90.68% 진행
[3524 / 3885] 	 90.71% 진행
[3525 / 3885] 	 90.73% 진행
[3526 / 3885] 	 90.76% 진행
[3527 / 3885] 	 90.79% 진행
[3528 / 3885] 	 90.81% 진행
[3529 / 3885] 	 90.84% 진행
[3530 / 3885] 	 90.86% 진행
[3531 / 3885] 	 90.89% 진행
[3532 / 3885] 	 90.91% 진행
[3533 / 3885] 	 90.94% 진행
[3534 / 3885] 	 90.97% 진행
[3535 / 3885] 	 90.99% 진행
[3536 / 3885] 	 91.02% 진행
[3537 / 3885] 	 91.04% 진행
[3538 / 3885] 	 91.07% 진행
[3539 / 3885

In [ ]:
if len(err_idx) != 0:
    re_err_idx = []

    for i in err_idx:
        try:
            all_results[i] = dict()

            link = news_links[i]
            
            # 실제 네이버 뉴스 웹페이지로 이동
            driver.get(link)

            # 페이지 로딩 대기
            time.sleep(0.6) 

            # 제목 추출하기
            title = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/p')
            title = title[0].text 

            # 본문 추출하기
            body = driver.find_elements(By.CLASS_NAME, 'article_content')
            body = body[0].text.replace('\n', '') 

            # 날짜 추출하기
            pubdate_element = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[1]/div[3]/div[1]/div[1]/div[2]')
            pubdate = pubdate_element[0].text
            pubdate = pubdate[5:24]

            all_results[i]['link'] = link
            all_results[i]['pubdate'] = pubdate
            all_results[i]['title'] = title
            all_results[i]['body'] = body

            # 진행 상황 확인용 코드
            print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행")    
            i += 1

            time.sleep(random.uniform(0.8, 1.5))

        except:
            print('오류가 발생했습니다.')
            re_err_idx.append(i)

            print(f"[{i+1} / {len(news_links)}] \t {(i+1)/len(news_links)*100:.2f}% 진행 - 오류 발생")    
            i += 1

            time.sleep(random.uniform(0.8, 1.5))

    print(len(all_results))
    print(re_err_idx)

[777 / 3885] 	 20.00% 진행
[778 / 3885] 	 20.03% 진행
[779 / 3885] 	 20.05% 진행
[780 / 3885] 	 20.08% 진행
[781 / 3885] 	 20.10% 진행
[782 / 3885] 	 20.13% 진행
[783 / 3885] 	 20.15% 진행
[784 / 3885] 	 20.18% 진행
[785 / 3885] 	 20.21% 진행
[3463 / 3885] 	 89.14% 진행
[3464 / 3885] 	 89.16% 진행
[3465 / 3885] 	 89.19% 진행
[3466 / 3885] 	 89.21% 진행
[3467 / 3885] 	 89.24% 진행
[3468 / 3885] 	 89.27% 진행
[3469 / 3885] 	 89.29% 진행
[3470 / 3885] 	 89.32% 진행
[3471 / 3885] 	 89.34% 진행
[3472 / 3885] 	 89.37% 진행
[3473 / 3885] 	 89.40% 진행
[3474 / 3885] 	 89.42% 진행
[3475 / 3885] 	 89.45% 진행
[3476 / 3885] 	 89.47% 진행
[3477 / 3885] 	 89.50% 진행
3885
[]


### 개별 기사 수집 끝 / datrame으로 저장

In [ ]:
# 수집한 정보들을 dataframe으로 변환
df = pd.DataFrame(all_results).T
df

,link,pubdate,title,body
0,https://www.busan.com/view/busan/view.php?code...,2022-01-05 10:38:05,"밀키트 판매 1위는 '양식'…마이셰프 분석, 고기류 판매 최다","마이셰프가 지난 한 해 판매 데이터를 통해 밀키트 트렌드(사진)를 결산하고, 올해 ..."
1,https://www.busan.com/view/busan/view.php?code...,2022-01-07 11:44:43,[유럽 인문학 기행] 베르사유 궁전에 흐르는 마리 앙투아네트의 사랑,[유럽 인문학 기행-프랑스] 베르사유 궁전(2)베르사유 궁전.■호프부르크의 마리아 ...
2,https://www.busan.com/view/busan/view.php?code...,2022-01-07 15:00:04,"수민동 새마을금고, 취약계층 1인 가구에 전기매트 지원",동래구 수민동 새마을금고(이사장 직무대행 강형기)는 지난 6일 수민동 행정복지센터(...
3,https://www.busan.com/view/busan/view.php?code...,2022-01-11 14:01:59,"남구, 쾌적한 생활환경을 위한 미세먼지 차단숲 조성","황령터널 ~ 대남교차로 구간 미세먼지 차단숲 6,020㎡ 조성부산시 남구가 황령터널..."
4,https://www.busan.com/view/busan/view.php?code...,2022-01-13 18:56:45,"아이가 ‘호’ 하고 불어 주자, 눈아이가 눈물을 흘렸다",눈아이/안녕달안녕달 작가는 그림책에서 눈아이가 아이의 따뜻한 위로에 눈물 흘리는 장...
...,...,...,...,...
3880,https://www.busan.com/view/busan/view.php?code...,2024-12-26 09:48:12,"'연인 폭행' 황철순 ""23kg 빠져 배만 볼록… 맨몸 운동도 금지라 사형선고와 같아""",보디빌더 황철순. 유튜브 영상 갈무리코미디 프로그램에서 징을 치는 '징맨'으로 이름...
3881,https://www.busan.com/view/busan/view.php?code...,2024-12-26 18:12:23,[부산일보 선정 2024년 10대 뉴스-국내] 대한민국 충격 빠뜨린 계엄,"1. 윤 대통령 비상계엄 선포와 해제, 탄핵소추연말 발생한 비상계엄 사태는 대한민국..."
3882,https://www.busan.com/view/busan/view.php?code...,2024-12-30 17:24:02,"[젊어지는 이야기] 갑상선 호르몬, 항노화에 도움?",김미경 해운대백병원 내분비내과 교수/동남권항노화의학회 사무총장2024년 12월 23...
3883,https://www.busan.com/view/busan/view.php?code...,2024-12-30 18:53:06,올해 최고 철도서비스 ‘실시간 열차 위치안내’ 1위 선정,코레일 베스트 서비스 투표 결과 발표여름철 KTX 무지연 청룡 운행 등 이어올해 최...


In [ ]:
# 수집한 기사들 중 중복인 경우 이를 제거
df_no_duplicates = df.drop_duplicates().reset_index(drop=True)
df_no_duplicates

,link,pubdate,title,body
0,https://www.busan.com/view/busan/view.php?code...,2022-01-05 10:38:05,"밀키트 판매 1위는 '양식'…마이셰프 분석, 고기류 판매 최다","마이셰프가 지난 한 해 판매 데이터를 통해 밀키트 트렌드(사진)를 결산하고, 올해 ..."
1,https://www.busan.com/view/busan/view.php?code...,2022-01-07 11:44:43,[유럽 인문학 기행] 베르사유 궁전에 흐르는 마리 앙투아네트의 사랑,[유럽 인문학 기행-프랑스] 베르사유 궁전(2)베르사유 궁전.■호프부르크의 마리아 ...
2,https://www.busan.com/view/busan/view.php?code...,2022-01-07 15:00:04,"수민동 새마을금고, 취약계층 1인 가구에 전기매트 지원",동래구 수민동 새마을금고(이사장 직무대행 강형기)는 지난 6일 수민동 행정복지센터(...
3,https://www.busan.com/view/busan/view.php?code...,2022-01-11 14:01:59,"남구, 쾌적한 생활환경을 위한 미세먼지 차단숲 조성","황령터널 ~ 대남교차로 구간 미세먼지 차단숲 6,020㎡ 조성부산시 남구가 황령터널..."
4,https://www.busan.com/view/busan/view.php?code...,2022-01-13 18:56:45,"아이가 ‘호’ 하고 불어 주자, 눈아이가 눈물을 흘렸다",눈아이/안녕달안녕달 작가는 그림책에서 눈아이가 아이의 따뜻한 위로에 눈물 흘리는 장...
...,...,...,...,...
3869,https://www.busan.com/view/busan/view.php?code...,2024-12-26 09:48:12,"'연인 폭행' 황철순 ""23kg 빠져 배만 볼록… 맨몸 운동도 금지라 사형선고와 같아""",보디빌더 황철순. 유튜브 영상 갈무리코미디 프로그램에서 징을 치는 '징맨'으로 이름...
3870,https://www.busan.com/view/busan/view.php?code...,2024-12-26 18:12:23,[부산일보 선정 2024년 10대 뉴스-국내] 대한민국 충격 빠뜨린 계엄,"1. 윤 대통령 비상계엄 선포와 해제, 탄핵소추연말 발생한 비상계엄 사태는 대한민국..."
3871,https://www.busan.com/view/busan/view.php?code...,2024-12-30 17:24:02,"[젊어지는 이야기] 갑상선 호르몬, 항노화에 도움?",김미경 해운대백병원 내분비내과 교수/동남권항노화의학회 사무총장2024년 12월 23...
3872,https://www.busan.com/view/busan/view.php?code...,2024-12-30 18:53:06,올해 최고 철도서비스 ‘실시간 열차 위치안내’ 1위 선정,코레일 베스트 서비스 투표 결과 발표여름철 KTX 무지연 청룡 운행 등 이어올해 최...


In [ ]:
# 중복인 기사 확인
df_duplicates = df[df.duplicated(keep=False)]
df_duplicates

,link,pubdate,title,body
1608,https://www.busan.com/view/busan/view.php?code...,2023-08-01 15:01:50,"[르포] 펄펄 끓는 물속, 애끓는 어민 속… “이러다 다 죽는다”",폭염 덮친 통영 양식장 가보니진해만 고수온 주의보→경보 격상양식 어류 2~3일 노출...
1609,https://www.busan.com/view/busan/view.php?code...,2023-08-01 15:01:50,"[르포] 펄펄 끓는 물속, 애끓는 어민 속… “이러다 다 죽는다”",폭염 덮친 통영 양식장 가보니진해만 고수온 주의보→경보 격상양식 어류 2~3일 노출...
2303,https://www.busan.com/view/busan/view.php?code...,2024-05-21 11:00:00,"부산 해운대·송정 해수욕장, 6월 1일 조기 개장…사전 현장점검","부산 7개 해수욕장 모두 전면 개장은 7월 1일해수부, 해수욕장 개장 전 사전점검…..."
2304,https://www.busan.com/view/busan/view.php?code...,2024-05-21 11:00:00,"부산 해운대·송정 해수욕장, 6월 1일 조기 개장…사전 현장점검","부산 7개 해수욕장 모두 전면 개장은 7월 1일해수부, 해수욕장 개장 전 사전점검…..."
2305,https://www.busan.com/view/busan/view.php?code...,2024-05-21 14:42:55,"안방서 1·2위 만나는 롯데, 탈꼴찌 넘어 중위권 도약 최대 고비",이번 주 KIA·삼성과 사직 6경기상위팀 상대 반등 계기 마련해야전준우·정훈 등 부...
2306,https://www.busan.com/view/busan/view.php?code...,2024-05-21 14:42:55,"안방서 1·2위 만나는 롯데, 탈꼴찌 넘어 중위권 도약 최대 고비",이번 주 KIA·삼성과 사직 6경기상위팀 상대 반등 계기 마련해야전준우·정훈 등 부...
2307,https://www.busan.com/view/busan/view.php?code...,2024-05-21 17:12:59,"대청동 지사협, ‘바람 솔솔~ 여름나기 지원’... 여름 이불․베개 전달","중구 대청동지역사회보장협의체(공동위원장 윤영숙․김지영, 이하 지사협)는 지난 20일..."
2308,https://www.busan.com/view/busan/view.php?code...,2024-05-21 17:12:59,"대청동 지사협, ‘바람 솔솔~ 여름나기 지원’... 여름 이불․베개 전달","중구 대청동지역사회보장협의체(공동위원장 윤영숙․김지영, 이하 지사협)는 지난 20일..."
2309,https://www.busan.com/view/busan/view.php?code...,2024-05-21 18:14:29,"[포토뉴스] 벌써 여름, 양산 '불티'",롯데백화점 부산본점 지하 1층 ‘아테스토니’ 매장에서는 다양한 양산 제품을 선보이고...
2310,https://www.busan.com/view/busan/view.php?code...,2024-05-21 18:14:29,"[포토뉴스] 벌써 여름, 양산 '불티'",롯데백화점 부산본점 지하 1층 ‘아테스토니’ 매장에서는 다양한 양산 제품을 선보이고...


In [ ]:
# 수집한 정보들을 csv로 저장 (이때 인코딩 형식은 utf8)
df_no_duplicates.to_csv(f'부산 뉴스.csv', index = False, encoding='utf-8-sig')